# Whisper Avanzado: Transcripción, Diarización y Post-procesado

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/voz-audio/01-whisper-avanzado.ipynb)

Transcripción profesional con chunking para archivos largos, timestamps de palabras y post-procesado con Claude.

In [ ]:
!pip install openai anthropic pydub faster-whisper -q

In [ ]:
import openai
import anthropic

openai_client = openai.OpenAI()
anthropic_client = anthropic.Anthropic()

## 1. Transcripción básica con Whisper API

In [ ]:
def transcribir_audio(ruta_audio: str, idioma: str = 'es') -> dict:
    """Transcripción básica con timestamps de segmentos."""
    with open(ruta_audio, 'rb') as f:
        respuesta = openai_client.audio.transcriptions.create(
            model='whisper-1',
            file=f,
            language=idioma,
            response_format='verbose_json',
            timestamp_granularities=['segment'],
        )
    return {
        'texto': respuesta.text,
        'idioma': respuesta.language,
        'duracion': respuesta.duration,
        'segmentos': [{'inicio': s.start, 'fin': s.end, 'texto': s.text} for s in (respuesta.segments or [])],
    }

# Demo con audio sintético (en entorno real, proporciona una ruta de audio real)
print('Función transcribir_audio() lista.')
print('Uso: resultado = transcribir_audio("mi_audio.mp3")')

## 2. Chunking para archivos > 25 MB

In [ ]:
from pydub import AudioSegment
import math

def chunkar_y_transcribir(ruta: str, max_mb: float = 20.0, idioma: str = 'es') -> str:
    """Divide un archivo grande y transcribe cada parte."""
    audio = AudioSegment.from_file(ruta)
    tam_mb = len(audio.raw_data) / (1024 * 1024)
    n_chunks = max(1, math.ceil(tam_mb / max_mb))
    duracion_chunk_ms = len(audio) // n_chunks
    
    transcripciones = []
    for i in range(n_chunks):
        inicio = i * duracion_chunk_ms
        fin = min((i + 1) * duracion_chunk_ms, len(audio))
        chunk = audio[inicio:fin]
        
        ruta_chunk = f'/tmp/chunk_{i:03d}.mp3'
        chunk.export(ruta_chunk, format='mp3', bitrate='64k')
        
        with open(ruta_chunk, 'rb') as f:
            resp = openai_client.audio.transcriptions.create(
                model='whisper-1', file=f, language=idioma
            )
        transcripciones.append(resp.text)
        print(f'  Chunk {i+1}/{n_chunks} transcrito')
    
    return ' '.join(transcripciones)

print('Función chunkar_y_transcribir() lista.')
print('Divide automáticamente archivos grandes y une las transcripciones.')

## 3. Post-procesado con Claude

In [ ]:
def limpiar_transcripcion(texto_crudo: str) -> str:
    """Limpia y añade puntuación a una transcripción cruda."""
    response = anthropic_client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=2048,
        messages=[{
            'role': 'user',
            'content': f"""Limpia esta transcripción de audio. Añade puntuación correcta,
corrige errores fonéticos obvios y separa en párrafos. No cambies el contenido.

TRANSCRIPCIÓN:
{texto_crudo}

TEXTO LIMPIO:"""
        }],
    )
    return response.content[0].text

def resumir_reunion(transcripcion: str) -> dict:
    """Extrae puntos clave y tareas de una transcripción de reunión."""
    response = anthropic_client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=1024,
        tools=[{
            'name': 'resumen_reunion',
            'description': 'Resumen estructurado de reunión',
            'input_schema': {
                'type': 'object',
                'properties': {
                    'resumen_ejecutivo': {'type': 'string'},
                    'puntos_clave': {'type': 'array', 'items': {'type': 'string'}},
                    'tareas': {
                        'type': 'array',
                        'items': {'type': 'object', 'properties': {
                            'tarea': {'type': 'string'},
                            'responsable': {'type': 'string'},
                        }},
                    },
                },
                'required': ['resumen_ejecutivo', 'puntos_clave', 'tareas'],
            },
        }],
        tool_choice={'type': 'tool', 'name': 'resumen_reunion'},
        messages=[{'role': 'user', 'content': f'Resume esta reunión:\n\n{transcripcion}'}],
    )
    return next(b for b in response.content if b.type == 'tool_use').input

# Demo con texto de ejemplo
transcripcion_ejemplo = """
buenas pues em empezamos la reunión de hoy el objetivo es revisar el avance del proyecto
de inteligencia artificial que estamos desarrollando para el cliente acuerdense que el
plazo es fin de mes entonces tenemos que acelerar maría puedes contarnos donde estamos
si entonces hemos terminado el módulo de extracción de datos y ahora estamos trabajando
en la integración con la api de claude necesitamos dos días más para terminarlo
"""

limpio = limpiar_transcripcion(transcripcion_ejemplo)
print('=== Transcripción limpia ===')
print(limpio)

resumen = resumir_reunion(limpio)
print('\n=== Resumen de reunión ===')
print(f'Resumen: {resumen["resumen_ejecutivo"]}')
print(f'Puntos: {resumen["puntos_clave"]}')
print(f'Tareas: {resumen["tareas"]}')

## 4. Generador de subtítulos SRT

In [ ]:
def generar_srt(segmentos: list[dict]) -> str:
    """Genera contenido SRT a partir de segmentos con timestamps."""
    def to_srt_time(s):
        h, m = int(s // 3600), int((s % 3600) // 60)
        sec, ms = int(s % 60), int((s % 1) * 1000)
        return f'{h:02d}:{m:02d}:{sec:02d},{ms:03d}'
    
    lineas = []
    for i, seg in enumerate(segmentos, 1):
        lineas += [str(i), f"{to_srt_time(seg['inicio'])} --> {to_srt_time(seg['fin'])}", seg['texto'].strip(), '']
    return '\n'.join(lineas)

# Ejemplo de segmentos
segmentos_ejemplo = [
    {'inicio': 0.0, 'fin': 3.5, 'texto': 'Bienvenidos a este tutorial de Inteligencia Artificial.'},
    {'inicio': 3.8, 'fin': 7.2, 'texto': 'Hoy vamos a aprender sobre transcripción de audio con Whisper.'},
    {'inicio': 7.5, 'fin': 11.0, 'texto': 'Esta tecnología permite convertir voz en texto de forma precisa.'},
]

srt = generar_srt(segmentos_ejemplo)
print(srt)